# 霍夫圓檢測_修正版.py
**電腦視覺與深度學習 › Day14 OpenCV 電腦視覺**  ·  ✅ 修正版

原本的霍夫圓檢測有一個藏得很深的 bug:圓心座標被轉成 uint16 之後直接相減,遇到負數會繞一圈變成很大的數,差距剛好 256 時距離會被算成 0,把不是圓的東西認成圓。

> 🎧 有逐段旁白、逐句拆解的版本在 [pyl Academy](https://friencdking0000.github.io/pyl-academy/#/learn/PY-CVAI/0/9)。
>
> **怎麼用**:由上往下,每格按左邊 ▶ 執行;或選單「執行階段 › 全部執行」。
>
> 📷 Colab 沒有攝影機、也開不了視窗:下面會**改用範例圖片**代替鏡頭,影像直接顯示在格子底下(只顯示前幾張)。

In [ ]:
#@title 0. 準備環境(下載課程範例檔、安裝套件)
!git clone -q --depth 1 https://github.com/friencdking0000/2026AI-STUST.git /content/2026AI-STUST 2>/dev/null; echo 課程檔案已就緒
%cd "/content/2026AI-STUST/Week1/DAY7"
import sys, os; sys.argv = sys.argv[:1]  # 讓 argparse / sys.argv 在 notebook 裡也正常
__file__ = os.path.join(os.getcwd(), '霍夫圓檢測.py')  # notebook 沒有 __file__,程式靠它找同資料夾的檔案

In [ ]:
#@title 1. Colab 沒有攝影機和視窗:改用範例圖片、影像直接顯示在下面
import cv2, numpy as np, urllib.request, os
from google.colab.patches import cv2_imshow
def _pyl_sample():
    img = np.full((480, 640, 3), 255, np.uint8)
    cv2.circle(img, (130, 140), 70, (0, 0, 255), -1)
    cv2.circle(img, (520, 360), 55, (255, 0, 0), -1)
    cv2.rectangle(img, (260, 60), (400, 200), (0, 160, 0), -1)
    cv2.fillPoly(img, [np.array([[90, 420], [230, 420], [160, 300]])], (0, 200, 255))
    cv2.fillPoly(img, [np.array([[300, 300], [430, 280], [450, 420], [320, 440]])], (200, 0, 200))
    return img
class _PylCam:
    def __init__(self, *a, **k): self.n = 0; self.img = _pyl_sample()
    def isOpened(self): return True
    def read(self):
        self.n += 1
        return (True, self.img.copy()) if self.n <= 30 else (False, None)
    def set(self, *a): return True
    def get(self, *a): return 0
    def release(self): pass
_pyl_VC = cv2.VideoCapture
cv2.VideoCapture = lambda src=0, *a, **k: _PylCam() if isinstance(src, int) else _pyl_VC(src, *a, **k)
_pyl_shown = [0]
def _pyl_imshow(name, img):
    _pyl_shown[0] += 1
    if _pyl_shown[0] <= 6:
        print('▼', name); cv2_imshow(img)
cv2.imshow = _pyl_imshow
cv2.waitKey = lambda *a: ord('q') if _pyl_shown[0] >= 6 else -1
cv2.destroyAllWindows = lambda *a: None
cv2.namedWindow = lambda *a, **k: None
cv2.createTrackbar = lambda *a, **k: None
cv2.getTrackbarPos = lambda *a, **k: 0
cv2.setMouseCallback = lambda *a, **k: None
cv2.resizeWindow = cv2.moveWindow = cv2.setWindowProperty = cv2.setWindowTitle = lambda *a, **k: None
_pyl_imread = cv2.imread
def _pyl_imread2(path, *a, **k):
    img = _pyl_imread(path, *a, **k)
    if img is None and not os.path.exists(str(path)):
        print('(找不到', path, ':課程 repo 沒附這張圖,改用範例圖片)')
        img = _pyl_sample()
    return img
cv2.imread = _pyl_imread2
print('已改用範例圖片代替攝影機')

### 修正版說明
開頭三行註解說明這是哪個檔案的修正版,原檔保留當錯誤示範,以及這次只修一個地方。

In [ ]:
# 修正版:對照原檔 霍夫圓檢測.py(原檔保留當「錯誤示範」)
# 修正 1:圓心座標被 np.uint16 轉過,直接和 int 相減會溢位繞回(差 256 時距離算成 0 → 誤判成圓);
#         改成先用 int() 轉回一般整數再相減。
import cv2
import numpy as np

# 1. 建立 VideoCapture 物件
cap = cv2.VideoCapture(0)

# 檢查鏡頭是否成功開啟
if not cap.isOpened():
    print("錯誤：無法開啟視訊鏡頭。")
    exit()

print("鏡頭已開啟！按下 'q' 鍵可結束程式。")

# 定義面積區間
min_area = 500   # 最小面積限制
max_area = 50000 # 最大面積限制 

### 先轉 int 再算距離
錯在哪:原本直接拿 uint16 的圓心和 cX 相減。為什麼會錯:uint16 不能有負數,十減兩百六十六會繞回變成六萬五千多,平方後又繞回零。怎麼修:用 int 把三個值先轉成一般整數。怎麼驗證:用一張只有方塊的假畫面,把假的霍夫圓心放在方塊中心右邊剛好 256 像素,原檔標成圓 1 個,修正版 0 個;圓心只差 3 像素時,兩版都正確標出圓。

In [ ]:
while True:
    # 2. 逐影格（Frame）讀取影像
    ret, frame = cap.read()
    
    # 如果讀取失敗，直接跳出迴圈
    if not ret:
        print("錯誤：無法接收影像畫面。")
        break

    img_copy = frame.copy()
    
    # 影像預處理：轉灰階 + 高斯模糊（霍夫圓對雜訊很敏感，模糊非常關鍵）
    gray_img = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray_img, (13, 13), 2)
    
    # A. 進行 Canny 與輪廓檢測 (用來篩選面積)
    edges = cv2.Canny(blurred, 50, 150)
    contours, hierarchy = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # B. 進行霍夫圓檢測
    # dp=1 (解析度), minDist=50 (圓心間的最小距離), param1=Canny高閾值, param2=圓心檢測閾值(越小越容易誤判)
    # minRadius/maxRadius 可根據實際大小調整，設為 0 代表不限制
    circles = cv2.HoughCircles(blurred, cv2.HOUGH_GRADIENT, dp=1, minDist=50,
                               param1=100, param2=35, minRadius=10, maxRadius=200)
    
    count = 0
    
    # 如果有偵測到任何霍夫圓，才開始與輪廓進行比對
    if circles is not None and len(contours) > 0:
        # 將圓的座標轉為整數
        circles = np.uint16(np.around(circles))
        
        for cnt in contours:
            # 1. 檢查面積條件
            area = cv2.contourArea(cnt)
            if min_area <= area <= max_area:
                
                # 2. 計算輪廓中心點 (Moments)
                M = cv2.moments(cnt)
                if M["m00"] != 0:
                    cX = int(M["m10"] / M["m00"])
                    cY = int(M["m01"] / M["m00"])
                    
                    # 3. 驗證這個輪廓附近有沒有對應的霍夫圓 (比對圓心距離)
                    is_circle = False
                    matched_radius = 0
                    
                    for circle in circles[0, :]:
                        # 【修正 1】circle 是 np.uint16(0~65535,不能有負數),
                        # 和 cX 相減若是負的會「繞回」變成很大的數;先轉成 int 再算
                        h_x, h_y, h_r = int(circle[0]), int(circle[1]), int(circle[2])
                        # 計算輪廓中心與霍夫圓心的歐幾里得距離
                        distance = np.sqrt((cX - h_x)**2 + (cY - h_y)**2)
                        
                        # 如果距離小於 15 像素，我們就認定這個輪廓「確實是個圓」
                        if distance < 15:
                            is_circle = True
                            matched_radius = h_r
                            break
                    
                    # 4. 條件完全滿足：在面積區間內，且是個圓 ➔ 繪製出來
                    if is_circle:
                        count += 1
                        
                        # 繪製輪廓 (紅色)
                        cv2.drawContours(img_copy, [cnt], -1, (0, 0, 255), 2)
                        
                        # 繪製圓心 (綠色點)
                        cv2.circle(img_copy, (cX, cY), 3, (0, 255, 0), -1)
                        
                        # 標註資訊
                        cv2.putText(img_copy, f"Circle #{count} R:{matched_radius}", (cX - 30, cY - 10), 
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)

    # 3. 顯示影像畫面
    cv2.imshow('Webcam', img_copy)
    # 可選：想觀察邊緣可以打開下一行
    # cv2.imshow('Edges', edges)

    # 4. 按下 'q' 鍵結束
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# 5. 釋放資源
cap.release()
cv2.destroyAllWindows()

## 小結
規則:拿 numpy 小型別的數字做減法或平方之前,先轉成 int 或 float。凸包檢測那支也有一模一樣的問題,下一個檔案一起修。

👉 回到 [pyl Academy 這堂課](https://friencdking0000.github.io/pyl-academy/#/learn/PY-CVAI/0/9) 看逐句拆解、做練習。